# Phase 6: 構造化RAG 完全評価

**目的**: 55件の階層化テストケース（L1-L5）での構造化RAG評価

**期待効果**:
- spatial_comparison: -8.9pt → +5pt以上
- advanced_comparison: -0.8pt → +10pt以上

**作成日**: 2026-01-22

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q tqdm pandas
print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認・メモリ管理
import torch
import gc

def print_memory():
    if torch.cuda.is_available():
        print(f"GPU VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")
    import psutil
    print(f"RAM: {psutil.virtual_memory().percent}%")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print_memory()

In [ ]:
# 1.3 Google Driveマウント・パス設定
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results"
CHROMA_DIR = f"{BASE_DIR}/chroma_db"

for d in [DATA_DIR, RESULTS_DIR, CHROMA_DIR]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.4 モデル設定
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
print(f"LLM: {LLM_MODEL}")
print(f"Embedding: {EMBEDDING_MODEL}")

## Section 2: データ・モジュール読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json
from collections import Counter

with open(f"{DATA_DIR}/poi_documents.json", "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

all_pois = []
for doc in poi_documents:
    if "metadata" in doc:
        poi = doc["metadata"].copy()
        poi["content"] = doc.get("content", "")
    else:
        poi = doc.copy()
    all_pois.append(poi)

print(f"POIデータ: {len(all_pois)}件")

In [ ]:
# 2.2 Phase 6モジュール読み込み・空間情報追加
from src.geo_utils import enrich_all_pois
from src.aggregator import compare_east_west, get_top_categories, analyze_category_by_direction
from src.structured_rag_system import analyze_question

enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

direction_counts = Counter(poi.get("direction_from_station", "不明") for poi in enriched_pois)
print(f"東側: {sum(direction_counts.get(d, 0) for d in ['east', 'northeast', 'southeast'])}件")
print(f"西側: {sum(direction_counts.get(d, 0) for d in ['west', 'northwest', 'southwest'])}件")

In [ ]:
# 2.3 テストケース読み込み
try:
    from src.test_cases_v2 import TEST_CASES_V2, get_test_cases_by_level, get_test_cases_by_subcategory
    print(f"テストケース読み込み完了: {len(TEST_CASES_V2)}件")
    
    # レベル別件数
    for level in ["L1", "L2", "L3", "L4", "L5"]:
        cases = get_test_cases_by_level(level)
        print(f"  {level}: {len(cases)}件")
except ImportError as e:
    print(f"テストケースモジュールが見つかりません: {e}")
    TEST_CASES_V2 = None

In [ ]:
# 2.4 評価関数読み込み
try:
    from src.evaluators_v2 import evaluate_response_v2, calculate_level_scores
    print("評価関数読み込み完了")
except ImportError:
    print("evaluators_v2.pyが見つかりません。基本評価を使用します。")
    evaluate_response_v2 = None

## Section 3: ベクトルストア・LLMセットアップ

In [ ]:
# 3.1 Embeddingモデルロード
from langchain_huggingface import HuggingFaceEmbeddings

print(f"Embeddingモデルロード中...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embeddingモデルロード完了")
print_memory()

In [ ]:
# 3.2 ベクトルストア読み込み/構築
from langchain_chroma import Chroma
from langchain_core.documents import Document

COLLECTION_NAME = "poi_shibuya_phase6"
chroma_exists = os.path.exists(f"{CHROMA_DIR}/{COLLECTION_NAME}")

if chroma_exists:
    print("既存ベクトルストア読み込み中...")
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings
    )
else:
    print("ベクトルストア構築中...")
    documents = []
    for poi in poi_documents:
        if "metadata" in poi:
            documents.append(Document(page_content=poi["content"], metadata=poi["metadata"]))
        else:
            content = poi.get("content", f"{poi.get('name', '')} - {poi.get('category', '')}")
            documents.append(Document(page_content=content, metadata=poi))
    
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR
    )
    print(f"構築完了: {len(documents)}件")

print_memory()

In [ ]:
# 3.3 Embeddingモデル解放
print("Embeddingモデル解放...")
del embeddings
clear_memory()
print_memory()

In [ ]:
# 3.4 LLMモデルロード
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"LLMモデルロード中: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("LLMモデルロード完了")
print_memory()

## Section 4: 構造化RAGシステム初期化

In [ ]:
# 4.1 構造化RAGシステムクラス定義
import time
from src.aggregator import filter_by_category

class StructuredRAGEvaluator:
    """構造化RAG評価用システム"""
    
    def __init__(self, model, tokenizer, vectorstore, all_pois):
        self.model = model
        self.tokenizer = tokenizer
        self.vectorstore = vectorstore
        self.all_pois = all_pois
        self.system_prompt = """あなたは渋谷エリアの地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
数値データがある場合は具体的な数字を使って回答してください。
情報がない場合は「情報がありません」と正直に回答してください。"""
    
    def _generate(self, prompt, max_tokens=512):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()
        return response
    
    def _build_context(self, question, analysis):
        """質問分析に基づいてコンテキストを構築"""
        context_parts = []
        
        # 東西比較が必要な場合
        if analysis.requires_comparison and "東" in question and "西" in question:
            cat = analysis.subcategories[0] if analysis.subcategories else None
            result = compare_east_west(self.all_pois, cat)
            context_parts.append(f"【東西比較】")
            context_parts.append(result.to_japanese())
            
            if cat:
                detail = analyze_category_by_direction(self.all_pois, cat)
                context_parts.append(f"\n方向別詳細:")
                for d, c in detail['by_direction'].items():
                    context_parts.append(f"  {d}: {c}件")
        
        # 集計が必要な場合
        elif analysis.requires_aggregation:
            if analysis.subcategories:
                cat = analysis.subcategories[0]
                filtered = filter_by_category(self.all_pois, cat)
                context_parts.append(f"【{cat}の集計】")
                context_parts.append(f"総数: {len(filtered)}件")
            else:
                top = get_top_categories(self.all_pois, 5)
                context_parts.append("【カテゴリランキング】")
                for i, cat in enumerate(top, 1):
                    context_parts.append(f"  {i}. {cat.category}: {cat.count}件")
        
        # ベクトル検索
        try:
            results = self.vectorstore.similarity_search(question, k=5)
            if results:
                context_parts.append("\n【関連POI情報】")
                for r in results:
                    name = r.metadata.get('name', '不明')
                    cat = r.metadata.get('category', '')
                    lat = r.metadata.get('lat', '')
                    lon = r.metadata.get('lon', '')
                    context_parts.append(f"- {name} ({cat})")
                    if lat and lon:
                        context_parts.append(f"  座標: ({lat}, {lon})")
        except Exception as e:
            print(f"ベクトル検索エラー: {e}")
        
        return "\n".join(context_parts)
    
    def query(self, question):
        """構造化RAGで質問に回答"""
        start = time.time()
        
        # 質問分析
        analysis = analyze_question(question)
        
        # コンテキスト構築
        context = self._build_context(question, analysis)
        
        # プロンプト構築
        prompt = f"""以下の情報を参考にして質問に回答してください。

{context}

【質問】
{question}

【回答】"""
        
        # 回答生成
        answer = self._generate(prompt)
        elapsed = time.time() - start
        
        return {
            "answer": answer,
            "analysis": analysis.to_dict(),
            "context": context,
            "time_sec": round(elapsed, 2)
        }
    
    def query_no_rag(self, question):
        """RAGなしで質問に回答（比較用）"""
        start = time.time()
        prompt = f"質問: {question}\n\n回答:"
        answer = self._generate(prompt)
        elapsed = time.time() - start
        return {"answer": answer, "time_sec": round(elapsed, 2)}

# システム初期化
rag_system = StructuredRAGEvaluator(model, tokenizer, vectorstore, enriched_pois)
print("構造化RAGシステム初期化完了")

## Section 5: 評価実行

In [ ]:
# 5.1 基本評価関数定義
def evaluate_response_basic(response, test_case):
    """基本的な評価関数"""
    answer = response.get("answer", "")
    
    # キーワードヒット率
    keywords = test_case.expected_keywords if hasattr(test_case, 'expected_keywords') else []
    if keywords:
        hits = sum(1 for kw in keywords if kw in answer)
        keyword_score = hits / len(keywords) * 100
    else:
        keyword_score = 50  # キーワードなしの場合は中間値
    
    # 座標含有
    has_coords = "35." in answer and "139." in answer
    coord_score = 100 if has_coords else 0
    
    # 数値含有（集計・比較質問用）
    import re
    numbers = re.findall(r'\d+', answer)
    has_numbers = len(numbers) > 0
    number_score = 100 if has_numbers else 0
    
    # 総合スコア
    total_score = (keyword_score * 0.4 + coord_score * 0.3 + number_score * 0.3)
    
    return {
        "keyword_score": keyword_score,
        "coord_score": coord_score,
        "number_score": number_score,
        "total_score": round(total_score, 1)
    }

In [ ]:
# 5.2 テスト実行関数
from tqdm import tqdm

def run_evaluation(rag_system, test_cases, use_rag=True, max_cases=None):
    """テストケースを実行して評価"""
    results = []
    
    cases_to_run = test_cases[:max_cases] if max_cases else test_cases
    
    for tc in tqdm(cases_to_run, desc="評価中"):
        try:
            # 回答生成
            if use_rag:
                response = rag_system.query(tc.prompt)
            else:
                response = rag_system.query_no_rag(tc.prompt)
            
            # 評価
            eval_result = evaluate_response_basic(response, tc)
            
            results.append({
                "id": tc.id,
                "level": tc.level,
                "category": tc.category,
                "subcategory": tc.subcategory,
                "prompt": tc.prompt,
                "answer": response["answer"][:500],
                "time_sec": response.get("time_sec", 0),
                "analysis": response.get("analysis", {}),
                "scores": eval_result
            })
            
            # メモリクリア
            clear_memory()
            
        except Exception as e:
            print(f"エラー ({tc.id}): {e}")
            results.append({
                "id": tc.id,
                "level": tc.level,
                "error": str(e)
            })
    
    return results

In [ ]:
# 5.3 spatial_comparison テストの実行
if TEST_CASES_V2:
    spatial_cases = get_test_cases_by_subcategory("spatial_comparison")
    print(f"spatial_comparison テスト: {len(spatial_cases)}件")
    
    spatial_results = run_evaluation(rag_system, spatial_cases, use_rag=True)
    
    print("\n=== spatial_comparison 結果 ===")
    for r in spatial_results:
        if "error" not in r:
            print(f"\n{r['id']}: {r['scores']['total_score']:.1f}pt")
            print(f"  質問: {r['prompt'][:50]}...")
            print(f"  回答: {r['answer'][:100]}...")

In [ ]:
# 5.4 全テスト実行（L1-L5）
if TEST_CASES_V2:
    print(f"全テストケース実行: {len(TEST_CASES_V2)}件")
    print("予想所要時間: 約30-40分")
    
    all_results_rag = run_evaluation(rag_system, TEST_CASES_V2, use_rag=True)
    print(f"\n完了: {len(all_results_rag)}件")

## Section 6: 結果分析

In [ ]:
# 6.1 レベル別スコア集計
import pandas as pd

def analyze_results(results):
    """結果を分析"""
    valid_results = [r for r in results if "error" not in r]
    
    # レベル別集計
    level_scores = {}
    for level in ["L1", "L2", "L3", "L4", "L5"]:
        level_results = [r for r in valid_results if r["level"] == level]
        if level_results:
            scores = [r["scores"]["total_score"] for r in level_results]
            level_scores[level] = {
                "count": len(scores),
                "avg": round(sum(scores) / len(scores), 1),
                "min": min(scores),
                "max": max(scores)
            }
    
    # サブカテゴリ別集計
    subcategory_scores = {}
    for r in valid_results:
        subcat = r["subcategory"]
        if subcat not in subcategory_scores:
            subcategory_scores[subcat] = []
        subcategory_scores[subcat].append(r["scores"]["total_score"])
    
    subcategory_avg = {}
    for subcat, scores in subcategory_scores.items():
        subcategory_avg[subcat] = round(sum(scores) / len(scores), 1)
    
    # 全体統計
    all_scores = [r["scores"]["total_score"] for r in valid_results]
    overall = {
        "count": len(all_scores),
        "avg": round(sum(all_scores) / len(all_scores), 1) if all_scores else 0,
        "min": min(all_scores) if all_scores else 0,
        "max": max(all_scores) if all_scores else 0
    }
    
    # 平均処理時間
    times = [r["time_sec"] for r in valid_results if "time_sec" in r]
    avg_time = round(sum(times) / len(times), 1) if times else 0
    
    return {
        "overall": overall,
        "by_level": level_scores,
        "by_subcategory": subcategory_avg,
        "avg_time_sec": avg_time
    }

if 'all_results_rag' in dir():
    analysis = analyze_results(all_results_rag)
    
    print("=" * 60)
    print("Phase 6 構造化RAG 評価結果")
    print("=" * 60)
    
    print(f"\n【全体】")
    print(f"  テスト数: {analysis['overall']['count']}件")
    print(f"  平均スコア: {analysis['overall']['avg']}pt")
    print(f"  平均処理時間: {analysis['avg_time_sec']}秒")
    
    print(f"\n【レベル別】")
    for level, stats in analysis['by_level'].items():
        print(f"  {level}: {stats['avg']}pt (n={stats['count']})")
    
    print(f"\n【サブカテゴリ別】")
    for subcat, avg in sorted(analysis['by_subcategory'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {subcat}: {avg}pt")

In [ ]:
# 6.2 Phase 5との比較
# Phase 5結果（COLAB_PROJECT_HANDOVER.mdより）
phase5_subcategory_scores = {
    "basic_location": 71.7,
    "basic_category": 58.3,
    "spatial_proximity": 61.7,
    "spatial_density": 65.0,
    "spatial_comparison": 51.4,  # 改善対象
    "constraint_single": 53.3,
    "constraint_multi": 53.3,
    "decision_location": 63.3,
    "decision_business": 63.3,
    "advanced_sensitivity": 60.0,
    "advanced_comparison": 59.5,  # 改善対象
    "advanced_uncertainty": 66.7
}

if 'analysis' in dir():
    print("\n【Phase 5 vs Phase 6 比較】")
    print(f"{'サブカテゴリ':<25} {'Phase5':>10} {'Phase6':>10} {'改善':>10}")
    print("-" * 60)
    
    for subcat, p5_score in phase5_subcategory_scores.items():
        p6_score = analysis['by_subcategory'].get(subcat, 0)
        diff = p6_score - p5_score
        marker = "✅" if diff > 0 else "❌" if diff < 0 else "-"
        print(f"{subcat:<25} {p5_score:>10.1f} {p6_score:>10.1f} {diff:>+10.1f} {marker}")

## Section 7: 結果保存

In [ ]:
# 7.1 結果をJSONで保存
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if 'all_results_rag' in dir() and 'analysis' in dir():
    result_data = {
        "timestamp": timestamp,
        "phase": "Phase 6 - Structured RAG Full Evaluation",
        "model": LLM_MODEL,
        "poi_count": len(enriched_pois),
        "test_count": len(all_results_rag),
        "analysis": analysis,
        "phase5_comparison": phase5_subcategory_scores,
        "results": all_results_rag
    }
    
    output_path = f"{RESULTS_DIR}/phase6_full_eval_{timestamp}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)
    
    print(f"結果保存: {output_path}")

In [ ]:
# 7.2 サマリーレポート生成
if 'analysis' in dir():
    report = f"""# Phase 6 構造化RAG 評価レポート

**実行日時**: {timestamp}
**モデル**: {LLM_MODEL}
**POI数**: {len(enriched_pois)}件
**テスト数**: {analysis['overall']['count']}件

## 全体結果

| 指標 | 値 |
|------|----|
| 平均スコア | {analysis['overall']['avg']}pt |
| 最小スコア | {analysis['overall']['min']}pt |
| 最大スコア | {analysis['overall']['max']}pt |
| 平均処理時間 | {analysis['avg_time_sec']}秒 |

## レベル別結果

| レベル | 平均スコア | テスト数 |
|--------|-----------|----------|
"""
    
    for level, stats in analysis['by_level'].items():
        report += f"| {level} | {stats['avg']}pt | {stats['count']}件 |\n"
    
    report += f"""
## 改善対象サブカテゴリ

| サブカテゴリ | Phase5 | Phase6 | 改善 |
|-------------|--------|--------|------|
| spatial_comparison | 51.4pt | {analysis['by_subcategory'].get('spatial_comparison', 0)}pt | {analysis['by_subcategory'].get('spatial_comparison', 0) - 51.4:+.1f}pt |
| advanced_comparison | 59.5pt | {analysis['by_subcategory'].get('advanced_comparison', 0)}pt | {analysis['by_subcategory'].get('advanced_comparison', 0) - 59.5:+.1f}pt |

## 次のステップ

1. グラフRAG実装（Phase 6.3）
2. ハイブリッドRAG統合（Phase 6.4）
3. ファインチューニング検討（Phase 7）
"""
    
    report_path = f"{RESULTS_DIR}/phase6_report_{timestamp}.md"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)
    
    print(f"レポート保存: {report_path}")
    print("\n" + report)